# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:\n  ", getattr(metadata, 'name', None))
print("\nDataset Description:\n  ", getattr(metadata, 'description', None))

## 2. Data Overview
List all available record sets (`@id`), with a preview of their fields and columns. All references use their Croissant `@id` attributes.

In [ ]:
# Helper function to obtain all record sets and their details
def list_record_sets(ds):
    recsets = ds.record_sets
    if not recsets:
        print('No record sets defined in this dataset.')
        return []

    data = []
    for recset in recsets:
        rs_id = getattr(recset, '@id', None)
        rs_name = getattr(recset, 'name', None)
        fields = getattr(recset, 'fields', [])
        field_ids = []
        for f in fields:
            field_id = getattr(f, '@id', None)
            field_name = getattr(f, 'name', None)
            field_ids.append({'@id': field_id, 'name': field_name})
        data.append({'record_set_id': rs_id, 'record_set_name': rs_name, 'fields': field_ids})
    return data

# List all record sets (@id and their fields' @id)
record_set_info = list_record_sets(dataset)
if record_set_info:
    for rs in record_set_info:
        print(f"Record Set: {rs['record_set_name']} (@id: {rs['record_set_id']})")
        if rs['fields']:
            print("  Fields:")
            for field in rs['fields']:
                print(f"    - {field['name']} (@id: {field['@id']})")
        else:
            print("  No fields defined.")
        print()
else:
    print("No record sets are available for exploration in this dataset.")

## 3. Data Extraction
Load data from each available record set into a DataFrame for easy analysis. All record sets and fields are referenced by their Croissant `@id` values.

In [ ]:
# Prepare a mapping from record set @id to DataFrame
dataframes = {}
record_set_ids = [rs['record_set_id'] for rs in record_set_info]

for record_set_id in record_set_ids:
    # Load records for this record_set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set: {record_set_id}")
        print(f"Columns (@id): {list(df.columns)}\n")
    else:
        print(f"No records found for record set: {record_set_id}\n")

# Show preview of one (the first) record set, if available
if dataframes:
    first_rs = next(iter(dataframes))
    print(f"\nPreview of first record set ({first_rs}):")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
- Filter records based on a chosen numeric field (must use the Croissant `@id`).
- Normalize the field.
- Optionally group the result by a categorical/group field via its `@id`.

**Note:** If no numeric field is available, this cell demonstrates the steps with mock IDs—replace as needed after reviewing columns above.

In [ ]:
# Select record set to explore (use the first one if any)
if dataframes:
    # Pick first record set and identify numeric field by @id
    rs_id = next(iter(dataframes))
    df = dataframes[rs_id]
    print(f"Available columns in record set ({rs_id}):\n{list(df.columns)}\n")
    
    # Attempt to auto-detect numeric fields to demonstrate (fallback: user provides @id)
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Auto-selected numeric field: {numeric_field_id}\n")
    else:
        # Place a placeholder if none found
        numeric_field_id = None
        print("No numeric field found. Please specify a valid numeric field `@id` for EDA.")

    # Demonstration filter and normalization if a field is found
    if numeric_field_id:
        threshold = df[numeric_field_id].mean()  # use mean as a reasonable threshold
        print(f"Filtering rows where {numeric_field_id} > {threshold:.2f}")
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:\n")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a non-numeric field
        group_candidates = [c for c in filtered_df.columns if filtered_df[c].dtype == 'object' and c != numeric_field_id]
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped.head())
        else:
            print("No suitable categorical field to group by found.")
else:
    print("No dataframes available for EDA. Please rerun previous steps or check the dataset.")

## 5. Visualization
Visualize the distribution of the filtered and normalized numeric field, or show relationships with a group/categorical field as found above. All references use Croissant `@id`.

**Note:** This example assumes a numeric field and group field are available. Modify field IDs as per your data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the normalized distribution if available
if 'filtered_df' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[norm_col], kde=True, bins=20)
    plt.title(f"Distribution of normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Count")
    plt.show()

    # If grouping was possible, plot mean statistic by group
    if 'group_field_id' in locals():
        grouped_reset = grouped.reset_index()
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_reset, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load and inspect a FAIR² Croissant-annotated dataset via its schema URL, 
- Enumerate its record sets, fields, and columns by their `@id`s,
- Extract records into pandas DataFrames, 
- Perform exploratory data analysis and normalization on selected numeric fields, and
- Visualize field distributions and grouped statistics referenced by Croissant `@id`s.

Replace field and record set IDs in code cells with your dataset's actual `@id` values (see Data Overview section) for further customized exploration.